# 🧠 Part 3: Uplift Model Training & Validation (T-Learner)
In this notebook, we train a T-Learner Uplift Model on the pilot campaign data and evaluate its performance using Qini Curves.

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import load_data
from src.features import compute_rfm_features
from src.uplift_model import TLearnerUpliftModel, calculate_qini_curve

## 1. Prepare Dataset

In [ ]:
dfs = load_data('../data')
features_df = compute_rfm_features(dfs['transactions'], dfs['customers'])
train_data = dfs['campaign'].merge(features_df, on='customer_id', how='inner')

feature_cols = [
    'recency_days', 'frequency_30d', 'monetary_90d', 
    'total_spend', 'total_visits', 'total_items', 
    'avg_basket_value', 'promo_ratio', 'customer_segment_code'
]
X = train_data[feature_cols]
treatment = train_data['is_treatment']
y = train_data['bought_after_promo']
print('Features shape:', X.shape)

## 2. Train Uplift Model

In [ ]:
model = TLearnerUpliftModel(use_lgbm=True)
model.fit(X, treatment, y)
print('T-Learner Uplift Model trained successfully!')

## 3. Predict & Calculate Qini Curve

In [ ]:
p_t, p_c, uplift = model.predict(X)
qini_df = calculate_qini_curve(y, treatment, uplift)

plt.figure(figsize=(8, 6))
plt.plot(qini_df['n_pop'], qini_df['qini'], label='T-Learner Model', color='indigo', lw=2)
plt.plot(qini_df['n_pop'], qini_df['random'], label='Random Baseline', color='grey', linestyle='--')
plt.title('Qini Curve (Pilot Campaign Evaluation)')
plt.xlabel('Number of Customers Targeted (Ranked by Uplift)')
plt.ylabel('Cumulative Incremental Conversions')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()